# 6장 · HTTP 요청과 JSON 응답 처리

HTTP 요청에서는 경로가 대상을 지정하고 상태 코드가 처리 결과를 알려 줍니다. 성공 응답을 받아도 본문이 예상한 JSON인지 따로 확인해야 합니다.

할 일 1번을 요청했는데 제목 대신 ‘찾을 수 없음’이 돌아왔습니다. 먼저 HTTP 응답을 받은 것인지, 받은 데이터가 원하는 기록인지부터 나눠서 확인합니다. 가상 응답으로 연습하므로 외부 서버와 계정은 필요하지 않습니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 httpx 0.28.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 경로 매개변수와 요청 ID

GET은 값을 읽어 달라는 요청입니다. /todos/1에서 /todos는 자료 종류, 마지막 1은 요청한 기록 번호입니다. 서버는 상태 코드와 본문을 돌려줍니다.


### 셀 01 · f-string으로 경로 구성

입력받은 번호를 주소에서 한 번만 사용하도록 만듭니다.


In [1]:
todo_id = 1
path = f"/todos/{todo_id}"
print(path)

/todos/1


/todos/1입니다. 뒤에서 받은 데이터의 id가 1인지 다시 확인할 때 같은 todo_id를 사용합니다.


### 셀 02 · JSON 응답 디코딩

서버가 보냈다고 가정한 JSON 한 건을 Python 값으로 바꿉니다.


In [2]:
import json
body = '{"id":1,"title":"  HTTP 연습  ","completed":false}'
todo = json.loads(body)
print(todo["title"].strip())

HTTP 연습


HTTP 연습이 나옵니다. 문자열을 해석했다고 해서 값까지 올바르다는 뜻은 아닙니다. 예를 들어 id가 2이면 요청한 기록과 다릅니다.


### 과제 03 · 요청 ID와 응답 ID 대조

응답에서 id만 2로 바꿨습니다. 받은 번호가 요청 번호와 같은지 출력하세요.


In [3]:
received = {**todo, "id": 2}
# 받은 번호와 요청한 번호를 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>received = {**todo, &quot;id&quot;: 2}
print(received[&quot;id&quot;] == todo_id)</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>False이므로 이 제목을 1번 기록으로 표시하면 안 됩니다. 상태 코드가 200인 경우에도 검사할 수 있는 내용입니다.</p>
</details>


## HTTP 상태 오류와 JSONDecodeError

404는 요청에 응답했지만 해당 자원을 찾지 못했다는 상태입니다. 타임아웃이면 읽을 응답 본문조차 없을 수 있습니다. 상태 확인을 먼저 하고 성공했을 때만 JSON을 읽습니다.


### 셀 04 · JSONDecodeError 처리

상태 코드는 200인데 본문에는 HTML이 담겼다고 가정합니다. 어떤 변환에서 실패하는지 확인하세요.


In [4]:
try:
    json.loads("<html>점검 중</html>")
except json.JSONDecodeError as error:
    print(type(error).__name__)

JSONDecodeError


JSONDecodeError가 나옵니다. URL을 다시 요청하기 전에 이미 받은 본문의 종류와 형식을 점검할 수 있습니다.


### 셀 05 · HTTPX의 상태 코드 예외

httpx가 설치된 수업 커널에서는 MockTransport가 정한 상태로 응답합니다. 네트워크 연결은 없습니다.

client.get(path)를 호출하면 MockTransport가 이 셀의 missing(request)를 대신 부릅니다. Response(404)를 만든 곳은 바로 그 함수입니다. 인터넷을 끊어도 이 404를 관찰할 수 있습니다.


In [5]:
from importlib.util import find_spec
if find_spec("httpx") is not None:
    import httpx
    def missing(request):
        return httpx.Response(404, json={"detail": "할 일을 찾을 수 없습니다."})
    with httpx.Client(transport=httpx.MockTransport(missing), base_url="https://example.invalid") as client:
        response = client.get(path)
        try:
            response.raise_for_status()
        except httpx.HTTPStatusError as error:
            print(error.response.status_code)
else:
    print("HTTPX가 설치된 커널에서 404 응답 실습을 실행하세요.")

404


HTTPX를 사용한 출력은 404입니다. 연결 실패 예외와 달리 error.response에서 서버의 상태 값을 읽습니다. 외부 서버의 가동 여부는 이 모의 요청으로 알 수 없습니다.


## JSON 불리언과 문자열 구분

JSON의 false는 파이썬의 False가 되지만 따옴표가 있는 "false"는 비어 있지 않은 문자열입니다. 자료형을 먼저 검사하면 잘못된 완료 표시를 막을 수 있습니다.


### 과제 06 · bool 타입 검사

잘못 들어온 문자열을 bool()로 바꾸지 말고 완료 값이 실제 bool인지 확인하세요.


In [6]:
completed = "false"
# completed의 자료형을 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>completed = &quot;false&quot;
print(type(completed) is bool)</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>False가 나옵니다. 응답 형식 오류로 기록할 입력입니다. bool(completed)를 쓰면 내용과 관계없이 비어 있지 않은 문자열이어서 True가 됩니다.</p>
</details>


## 돌아보기

주소를 잘못 썼을 때, 기다리는 동안 연결이 끊겼을 때, JSON의 칸이 잘못됐을 때를 차례로 떠올려 보세요. 같은 ‘조회 실패’ 화면에서도 기록할 정보와 다음에 확인할 곳이 달라집니다.

### 조회에서 사용까지

**GET /todos/1**　타임아웃을 설정하고 요청합니다.

```
요청할 id = 1
```

**성공 코드 검사**　raise_for_status()로 2xx가 아닌 응답을 구분합니다.

```
200 → 본문 읽기
404 → 상태 검사에서 중단 (셀 05)
```

**본문을 Python 값으로**　JSON 파싱이 실패할 수도 있습니다.

```
JSON → Python 값
HTML → JSON 변환에서 중단 (셀 04)
```

**필요한 키·타입 검사**　id·title·completed 계약을 확인한 뒤 사용합니다.

```
title = "HTTP 연습"
```
